# Victor Travel Intelligence — LoRA fine-tune

Base model: `Qwen/Qwen3-0.6B`. The goal of V1 is **a pipeline that runs end to end**, not a good model. A 0.6B model gives mediocre travel advice; that is expected and is not the thing being measured.

## Order of the notebook, and why it is this order

| Step | What | Why here |
|---|---|---|
| 1 | GPU + install | fail fast if the runtime has no GPU |
| 2 | Upload dataset | 3 files from `ai/dataset/` |
| 3 | **Baseline A and B** | **before training** — otherwise there is nothing to compare against |
| 4 | LoRA + SFT | the actual training |
| 5 | State C | same prompts, fine-tuned model |
| 6 | Compare A / B / C | side by side, by eye |
| 7 | Merge + download | a folder you can run on your own machine |

**Step 3 is the one people skip and the one that decides whether any of this was worth doing.** If B (base + system prompt) already produces what you want, the honest answer is that you do not need an adapter — a system prompt is free.

## The Qwen3 thinking trap

Qwen3 ships with a reasoning mode that is **on by default**. Measured against the Hugging Face router on 2026-08-22: every hosted Qwen3 emits a long `reasoning_content` block, and `enable_thinking: false` sent in the request is accepted and ignored.

Running locally is what gives that control back — `apply_chat_template(..., enable_thinking=False)` genuinely turns it off.

**Our dataset contains no reasoning blocks**, so training must use the non-thinking template. Mixing the two teaches the model neither. Every template call in this notebook passes `enable_thinking=False`, and that is deliberate rather than incidental.

## 1 — GPU and install

Runtime → Change runtime type → T4 GPU, if `nvidia-smi` shows nothing.

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q -U transformers datasets trl peft accelerate

# Colab ships torchao 0.10.0 preinstalled. Recent peft requires >0.16.0 and, on finding an
# older one, raises ImportError from is_torchao_available() rather than returning False - so
# constructing SFTTrainer dies with a message about a library this notebook never uses.
#
# Removed rather than upgraded: torchao is for quantization, LoRA in fp16 needs none of it,
# and upgrading it can drag torch itself along and break the runtime for real.
%pip uninstall -q -y torchao

import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 2 — Upload the dataset

**These four files live in two different folders, and Colab's file picker only opens one folder at a time.** Run the cell twice — once per folder — or the fourth file goes missing and the failure surfaces two cells later.

From `ai/dataset/`:

- `train.jsonl`
- `validation.jsonl`
- `test.jsonl`

From `ai/evaluation/`:

- `test_cases.json`

Generate the three `.jsonl` files first with `python build_dataset.py` — they already carry the system prompt, prepended from `backend/app/ai/prompts.py`.

In [ ]:
from pathlib import Path

from google.colab import files

REQUIRED = ["train.jsonl", "validation.jsonl", "test.jsonl", "test_cases.json"]

files.upload()

missing = [name for name in REQUIRED if not Path(name).exists()]
print()
for name in REQUIRED:
    print(("  ok      " if name not in missing else "  MISSING ") + name)

if missing:
    print()
    print("Run this cell again and pick the missing file(s).")
    print("The .jsonl files are in ai/dataset/, test_cases.json is in ai/evaluation/.")

In [ ]:
import json

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "validation.jsonl",
        "test": "test.jsonl",
    },
)
print(dataset)

# Look at one before training anything. A malformed dataset is far cheaper to catch here than
# after a training run that appears to succeed.
example = dataset["train"][0]
for turn in example["messages"]:
    print(f"--- {turn['role']} ---")
    print(turn["content"][:400])
    print()

In [ ]:
from pathlib import Path

if not Path("test_cases.json").exists():
    print("test_cases.json is not here yet - it lives in ai/evaluation/, not ai/dataset/.")
    from google.colab import files

    files.upload()

with open("test_cases.json", encoding="utf-8") as handle:
    TEST_CASES = json.load(handle)

# The system prompt is identical in every training example, so lifting it from the first one
# keeps this notebook honest: whatever the backend sends is what gets evaluated here.
SYSTEM_PROMPT = dataset["train"][0]["messages"][0]["content"]
assert dataset["train"][0]["messages"][0]["role"] == "system"

print(f"{len(TEST_CASES)} evaluation cases")
print(f"system prompt: {len(SYSTEM_PROMPT)} chars")

## 3 — Baseline: states A and B, **before** any training

| State | What |
|---|---|
| **A** | base model, no system prompt |
| **B** | base model + system prompt — what `/intel` runs today |

Record both. Without them there is no way to attribute any later improvement to the adapter rather than to the prompt.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)
print(model.config.model_type, sum(p.numel() for p in model.parameters()) / 1e6, "M params")

In [ ]:
import re

def generate(question, system_prompt=None, active_model=None, max_new_tokens=400):
    """One answer, with thinking mode explicitly off.

    `enable_thinking=False` is the whole reason this runs locally rather than through the
    router: hosted Qwen3 ignores the equivalent request field. Our dataset has no reasoning
    blocks, so training and evaluation must both use the non-thinking template or they teach
    different things.

    Two details that are easy to get wrong and produce garbage rather than an error:

    1. `add_special_tokens=False`. `apply_chat_template(tokenize=False)` already emitted the
       control tokens as text; tokenizing that string again with the default would let the
       tokenizer add its own on top, and the model sees a prompt shaped unlike anything it
       was trained on.
    2. Sampling values. Qwen3 documents different settings for thinking and non-thinking
       mode; these are the non-thinking ones. `repetition_penalty` is what stops a small
       model latching onto one token and repeating it down the page.
    """
    target = active_model if active_model is not None else model

    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": question})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(target.device)

    outputs = target.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = outputs[0][inputs["input_ids"].shape[-1] :]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # A model trained on the non-thinking template can reproduce the empty block in its
    # own output. Left in, it becomes the first line and every answer fails the heading
    # check for a reason unrelated to what is being measured.
    answer = re.sub(r"^<think>.*?</think>", "", answer, flags=re.DOTALL).strip()
    return answer


# Sanity check, and a real diagnostic: run the same question with and without the system prompt.
# A model that answers the bare question but produces nonsense under the system prompt is telling
# you the prompt is too heavy for its size, not that it is broken.
print("=== no system prompt ===")
print(generate("Name three cities in Japan."))
print()
print("=== with system prompt ===")
print(generate("Name three cities in Japan.", SYSTEM_PROMPT))


In [ ]:
results = {}


def sweep(state, system_prompt, active_model=None):
    answers = {}
    for case in TEST_CASES:
        answer = generate(case["input"], system_prompt, active_model)
        answers[case["id"]] = answer
        print(f"=== [{state}] {case['id']} ===")
        print(case["input"])
        print("-" * 60)
        print(answer)
        print()
    results[state] = answers
    return answers


sweep("A", None)

In [ ]:
sweep("B", SYSTEM_PROMPT)

**Stop and read B before continuing.**

Judge it on the four things the dataset is meant to teach:

1. Does it use the UPPERCASE heading format?
2. Does it **ask back** instead of guessing when the request is underspecified?
3. Does it refuse to invent live prices and opening hours?
4. Does it stay short — 5 to 12 lines — rather than drifting into paragraphs?

Whichever of these B fails is what the adapter has to fix, and is where extra training examples should go.

In [ ]:
# Save the baselines before training. A runtime restart - the usual cure for a dependency
# clash - wipes every variable, and re-running the A and B sweeps is the slowest part here.
import json

with open("results_ab.json", "w") as handle:
    json.dump(results, handle)
print("saved:", list(results))

# After a restart, reload with:
#   with open("results_ab.json") as handle:
#       results = json.load(handle)

## 4 — LoRA + SFT

The base model stays frozen; only the adapter trains. On a T4 with 20 examples this takes a couple of minutes.

The dataset is pre-formatted into a `text` field rather than handed to `SFTTrainer` as conversations, because that is the only way to be sure `enable_thinking=False` is applied — the trainer's own templating would use Qwen3's default, which is thinking-on, and the dataset would no longer match what evaluation and the backend do.

In [ ]:
def to_text(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                enable_thinking=False,
            )
            for messages in batch["messages"]
        ]
    }


formatted = dataset.map(to_text, batched=True, remove_columns=["messages"])

# Read one, and read it carefully - the check is not 'is there a think block'.
#
#   <think>

</think>   EMPTY  -> correct. This is how Qwen3 marks thinking as OFF:
#                                 the template pre-fills a closed, empty block so the
#                                 model has nothing to reason into.
#   <think>Okay, the user...</think>   -> WRONG. enable_thinking did not take effect,
#                                         and training would teach the reasoning shape.
#
# Also confirm the answer itself: labels should line up in one column.
print(formatted["train"][0]["text"][-800:])

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

# Epochs: 4, not 8, and the number came from a run rather than a guess.
#
# The first pass at 8 epochs on 24 examples showed training loss falling all the way from
# 4.29 to 0.38 while VALIDATION loss bottomed out at epoch 4 (1.426) and then climbed every
# epoch after: 1.476, 1.515, 1.569, 1.635. Mean token accuracy went flat over the same span.
# Everything after epoch 4 was memorisation of 24 examples, paid for in time.
#
# `load_best_model_at_end` makes this self-correcting rather than a number to keep tuning:
# checkpoints are kept per epoch and the one with the lowest eval_loss is what you end up
# holding, whatever the epoch count is set to.
training_args = SFTConfig(
    output_dir="./travel-intelligence",
    num_train_epochs=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    max_length=2048,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted["train"],
    eval_dataset=formatted["validation"],
    peft_config=peft_config,
)

trainer.train()

**A falling training loss is not evidence the model got better.** Watch the two columns against each other instead.

From the first real run of this notebook, 8 epochs on 24 examples:

| Epoch | Train | Validation |
|---|---|---|
| 3 | 1.620 | 1.475 |
| **4** | 1.308 | **1.426** |
| 5 | 0.849 | 1.476 |
| 8 | 0.382 | 1.635 |

Training loss kept falling for four more epochs while validation loss climbed the whole way. That gap is memorisation, and the point where the two part company — epoch 4 — is the model worth keeping. `load_best_model_at_end` above now picks it automatically.

With three validation examples the number is noisy; four consecutive rises is still a signal, and it agreed with what the answers looked like.

## 5 — State C, and the comparison

In [ ]:
sweep("C", SYSTEM_PROMPT, trainer.model)

In [ ]:
for case in TEST_CASES:
    case_id = case["id"]
    print("#" * 70)
    print(f"# {case_id}: {case['input']}")
    print("#" * 70)
    for behaviour in case["expected_behavior"]:
        print(f"  expected: {behaviour}")
    for state in ("A", "B", "C"):
        print()
        print(f"----- {state} -----")
        print(results.get(state, {}).get(case_id, "(not run)"))
    print()
    print()

### How to read this

| Outcome | What it means | What to do |
|---|---|---|
| C clearly better than B | the adapter earned its place | merge and ship it |
| C ≈ B | the system prompt was already doing the work | keep the prompt, skip the adapter — this is a real and common result |
| C worse than B | usually too few examples, or too many epochs on too few | more examples before more epochs |
| C repeats training answers verbatim | memorisation | more variety, fewer epochs |

The fourth row is the likeliest at 16 examples. It is not a failure of the pipeline — it is the pipeline working and telling you the dataset is small.

### Scoring, so the comparison is a number rather than an impression

Reading thirty answers by eye across three states is the fastest way to talk yourself into a result. These five checks are mechanical, and they are exactly what the dataset teaches — so if training worked, they move.

None of them scores travel *knowledge*. That is deliberate: fine-tuning teaches behaviour, and a 0.6B model will keep putting Kinkaku-ji in Tokyo whatever we do here.

In [ ]:
import re

OPENERS = {
    "TRAVEL ANALYSIS",
    "DESTINATION ANALYSIS",
    "COMPARISON",
    "ITINERARY",
    "BUDGET ESTIMATE",
    "PARAMETERS REQUIRED",
    "LIVE DATA REQUIRED",
    "OUT OF SCOPE",
}

# A price with a unit. Catches "¥25,000", "10-25 thousand yen", "$40 per night".
MONEY = re.compile(r"[¥$€£]\s?\d|\d[\d,.]*\s?(yen|usd|dollars|euros|thousand)", re.I)

# Assertions about the live world, including the hedged kind. "usually available" is still a claim.
ASSERTS = re.compile(
    r"\b(are available|is available|no flights|are no flights|will be on schedule"
    r"|is open|usually|typically costs|should be)\b",
    re.I,
)

# A label line: uppercase label, two or more spaces, then a value.
LABEL = re.compile(r"^[A-Z][A-Z0-9 ]*[A-Z0-9] {2,}\S")

MARKDOWN = re.compile(r"\*\*|^\s*[-*] |^#", re.M)

VALUE_COLUMN = 14


def score_answer(case_id: str, text: str) -> dict[str, bool]:
    """Five checks, plus one that depends on what the case is testing."""
    lines = [line.rstrip() for line in text.strip().splitlines()]
    body = [line for line in lines if line.strip()]

    checks: dict[str, bool] = {}
    checks["heading"] = bool(body) and body[0].strip() in OPENERS

    # `has_labels`, not "every value sits in column 15".
    #
    # The strict column check was dropped after measuring what it actually rejected. A fine-tuned
    # Qwen3-0.6B produced columns of 14, 15, 16 and 17 across cases - and in one answer, 14 on the
    # first line and 15 on the three below it, so the lines did not even align with each other.
    # Another reached for a TAB character. Counting spaces requires the model to reason about
    # whitespace the tokenizer merges away; it can only reproduce spacing it has memorised.
    #
    # It is also the wrong layer. `IntelConsole` renders in a monospace whitespace-pre-wrap block
    # and can align columns in CSS for free. Spending training data on something the stylesheet
    # does better is the definition of a badly chosen requirement.
    labels = [line for line in lines if LABEL.match(line)]
    checks["has_labels"] = bool(labels)

    checks["no_markdown"] = not MARKDOWN.search(text)
    checks["length"] = len(body) <= 12

    if case_id.startswith("clarify"):
        # The highest-priority behaviour: ask, and do not staple a guess underneath.
        others = {line.strip() for line in body if line.strip() in OPENERS} - {
            "PARAMETERS REQUIRED"
        }
        checks["asks_only"] = (
            bool(body) and body[0].strip() == "PARAMETERS REQUIRED" and not others
        )
    elif case_id.startswith("live-data"):
        checks["refuses"] = (
            bool(body)
            and body[0].strip() == "LIVE DATA REQUIRED"
            and not MONEY.search(text)
            and not ASSERTS.search(text)
        )

    return checks


def report(state: str, results: dict, test_cases: list) -> None:
    """Print a per-case grid and a per-check total for one state."""
    answers = results.get(state, {})
    if not answers:
        print(f"state {state}: not run")
        return

    totals: dict[str, int] = {}
    passed: dict[str, int] = {}

    print(f"===== STATE {state} =====")
    for case in test_cases:
        case_id = case["id"]
        checks = score_answer(case_id, answers.get(case_id, ""))
        marks = "".join("o" if value else "." for value in checks.values())
        failed = " ".join(name for name, value in checks.items() if not value)
        print(f"  {case_id:<14} {marks:<6}  {failed}")
        for name, value in checks.items():
            totals[name] = totals.get(name, 0) + 1
            passed[name] = passed.get(name, 0) + (1 if value else 0)

    print()
    for name in totals:
        print(f"  {name:<12} {passed[name]:>2}/{totals[name]}")
    print()


for state in ("A", "B", "C"):
    report(state, results, TEST_CASES)


## 6 — Merge and download, for running on your own machine

You chose to run the finished model locally, so the adapter is merged into the base weights here: one self-contained folder, no PEFT needed at inference time.

In [ ]:
merged = trainer.model.merge_and_unload()
merged.save_pretrained("victor-travel-intelligence")
tokenizer.save_pretrained("victor-travel-intelligence")

# The adapter on its own is kept too: a few MB rather than ~1.2 GB, and the thing to keep if you
# ever want to apply the same training to a different base model.
trainer.model.save_pretrained("victor-travel-intelligence-adapter")

!du -sh victor-travel-intelligence victor-travel-intelligence-adapter

In [ ]:
!zip -qr victor-travel-intelligence.zip victor-travel-intelligence

from google.colab import files

files.download("victor-travel-intelligence.zip")

## What happens next, back in the repo

Unzip the folder somewhere on your machine, then serve it locally. `backend/app/ai/client.py` speaks the OpenAI chat-completions shape, so any local server offering that endpoint drops straight in — point `HF_ROUTER_URL` at it and set `HF_MODEL` to the local model name.

Two things to remember when you get there:

1. **Keep `enable_thinking=False` on the serving side too.** Everything in this notebook assumes it, and a server that re-enables Qwen3's default will produce reasoning blocks the console renders as noise.
2. **The `/intel` console needs no code change.** That was the point of building the backend before the dataset.